# DASHI: Dataset shift analysis and characterization in python

#### David Fernández Narro, Pablo Ferri Borredà, Ángel Sanchez-García, Juan M. García-Gómez, Carlos Sáez

*dfernar@upv.es
**pabferb2@upv.es
***ansan12a@upv.es
****juanmig@upv.es
*****carsaesi@upv.es

---
##### Library: `dashi`

Welcome to the `dashi` Python library! This notebook demonstrates the key features and functionality of the library, as well as an example to help you to understand its usage and applications for **dataset shift characterization**.

<a id="introduction"></a>
## 1 Introduction

---

<a id="what-is-dashi"></a>
### 1.1 What is `dashi`?
dashi is a Python library for analyzing and characterizing temporal and multi-source dataset shifts. It offers unsupervised and supervised tools for quantifying and visualizing covariate, concept, and prior shifts, supporting trustworthy artificial intelligence development and evaluation.

#### Why is `dashi` important?
Dataset shifts—unexpected changes in data distribution over time or across sources—can significantly impact the performance of machine learning models. Such shifts violate the assumption that training and test data come from the same distribution, often leading to reduced model accuracy, calibration, and generalizability. `dashi` helps AI developers and managers detect and delineate these shifts while providing insights to mitigate their effects, ensuring robust and reliable model development and helping prevent model obsolescence.

<a id="key-features"></a>
### 1.2 Key Features
- **Shift scope:**
  - **Temporal**: Outlines changes in data over time, enabling analysis of trends, seasonality, and abrupt shifts across different periods or temporal batches. This scope requires data to be labelled with a date.
  - **Source/domain**: Allows analysis of differences between multiple data sources or domains, being useful for identifying variability, biases, or inconsistencies among data from different origins, such as hospitals, laboratories, or geographic regions. This scope requires data to be labelled with the source/domain, without any specific chronological order.


- **Types of dataset shifts:**
  - **Covariate**: Changes in the distribution of input variables (features), while the relationship between inputs and
  outputs remain the same.
  - **Prior**: Changes in the distribution of the target variable (labels or outcomes), while the
  conditional distribution of features given the label remains stable.
  - **Concept**: Changes in the relationship between input variables and the target variable, i.e., the
  conditional distribution of the target given the features changes or vice versa.


- **Unsupervised approach:**
Distribution-based, model-agnostic delineation and characterization of dataset shifts by analysing the data covariate
and outcome-conditional statistical distributions and projecting and visualizing their dissimilarities across the
temporal or source scope.
This process involves:
  - Estimating data statistical distributions across batches along the temporal or source scope.
  - Projecting these distributions onto non-parametric statistical manifolds based on different embedding functions
  including the Jensen-Shannon distance + Multi Dimensional Scaling and Principal Component Analysis.


- **Supervised approach:**
Model-based delineation and characterization of dataset shifts, by relying on automated generation of classification or
regression models trained on batched data across the selected scope (temporal or multi-source). This allows for the
detailed analysis of how dataset shifts impact model performance, helping to pinpoint areas of potential degradation.
This process involves:
  - Training classification or regression models using Random Forests across batches along the selected scope.
  - Calculating model contingency matrices pairwise across the batched models and evaluating multiple evaluation metrics.

  

<a id="installation"></a>
### 1.3 Installation
You can install `dashi` using pip:

```bash
pip install dashi
```

Or install from source:

```bash
git clone https://github.com/bdslab-upv/dashi
cd dashi
pip install .
```

In [ ]:
%pip install -U dashi

# 2 Case study: biomedical AI modelling
---
#### 2.1 Why biomedical data?
Biomedical data repositories and proprietary biomedical research databases are expanding rapidly, both in terms of sample size and the diversity of collected variables. This
growth is driven by the widespread adoption of data-sharing initiatives, advancements in technological infrastructures, and the continuous population of these repositories over
extended periods (Gewin, 2016; Andreu-Perez et al., 2015).

However, this increased availability of data presents significant challenges. The integration of data from diverse sources over time introduces potential issues
that can impede its reuse in research contexts, such as population studies or statistical and machine learning modeling. Differences in protocols, population characteristics,
and unforeseen biases, whether introduced by systems or human error, can lead to **temporal or multi-source dataset shifts**
(Quiñonero-Candela, 2009; Moreno-Torres et al., 2012). These shifts manifest as changes in statistical distributions, altering reference characteristics and potentially
degrading model performance. Addressing these issues is particularly critical for ensuring robust and reliable predictive modeling and population health studies, as temporal
shifts in electronic health records (EHRs) have been identified as a major concern (Sáez et al., 2020; Schlegel & Ficheur, 2017).

This variability underscores the importance of addressing **Data Quality (DQ)** as a critical factor in enabling the reliable reuse of biomedical data. By detecting,
understanding,
and mitigating dataset shifts, researchers can ensure the robustness and validity of their findings, even within the context of an evolving and diverse data landscape.

#### 2.2 Use case description
##### Gestational Diabetes Mellitus (GDM)
Gestational Diabetes Mellitus (GDM) is a condition characterized by glucose intolerance that is first recognized during pregnancy. It is associated with various adverse outcomes for both the mother and the baby, including an increased risk of developing type 2 diabetes later in life. The prevalence of GDM has been rising globally, making it a significant public health concern.
This tutorial is grounded in a realistic antenatal-clinic use case: using Oral Glucose Tolerance Test (OGTT) results and routine factors (fasting and 1-h glucose, HbA1c, BMI, maternal age) to predict GDM. The dataset contains simulated data from 96.000 patients collected in monthly batches between 2021 and 2024 from a single hospital. During the initial two years of the study period, a substantial overlap between gestational diabetes mellitus (GDM) and non-GDM cases was observed. This pattern can plausibly be explained by inconsistencies in clinical coding practices, particularly for postpartum diagnoses. In ICD-10, GDM is represented by multiple codes (e.g., O24.41x for pregnancy, O24.42x for childbirth, and O24.43x for the puerperium), with additional subcategories for treatment modality. When these codes are incorrectly applied or omitted—such as failing to assign postpartum codes (O24.43x) for patients previously diagnosed during pregnancy, the ground truth labels derived from electronic health records become unreliable. Such discrepancies often arise during transitions in coding systems or when staff training is incomplete. After audits and retraining initiatives, the coding accuracy improved gradually over the third year, and by the fourth year, the overlap between classes diminished, restoring label consistency.


##### Dataset's variables explanation:
- gdm_diagnosis: binary label indicating 1 = GDM, 0 = non-GDM
- fasting_glucose_mgdl: fasting glucose value before OGTT (mg/dL)
- oneh_glucose_mgdl: glucose value 1h after OGTT (mg/dL)
- hba1c_percent: First-trimester HbA1c (%)
- bmi_kg_m2: Pre-pregnancy BMI (kg/m^2)
- maternal_age_years: Maternal age in years
- batch_date: Date with month granularity


## 3 Data Loading
---
The first step is to read the file that contains the tabular data for the analysis. To do it, the user can apply the `read_csv` function from `pandas` library.
An example of how to read the CSV file is shown next:

In [ ]:
import pandas as pd

url = "https://media.githubusercontent.com/media/bdslab-upv/dashi/refs/heads/main/examples/gdm_dataset.csv"
data = pd.read_csv(url)

## 4 Format data
---

Once the data is loaded, the next step is to format the data. This data formatting consists of type hinting of date, categorical and numerical variables, as well as removing rows without date or source information. In `dashi`, we can do it with the `format_data` function, as shown next.

In [ ]:
import dashi as ds

# METADATA
DATE_COLUMN = 'batch_date'
LABEL = 'gdm_diagnosis'
PERIOD = 'month'
numerical_features = ['fasting_glucose_mgdl', 'oneh_glucose_mgdl', 'hba1c_percent',
                      'bmi_kg_m2', 'maternal_age_years']
categorical_features = [LABEL]

# Data formatting
dataset_formatted = ds.format_data(
    input_dataframe=data,
    date_column_name=DATE_COLUMN,
    numerical_column_names=numerical_features,
    categorical_column_names=categorical_features,
    date_format='%Y-%m-%d'
)

## 5 Unsupervised characterization
---
In this section, we will focus on unsupervised characterization of the dataset to assess the temporal evolution of the data, independent of any model-based assumptions. The goal is to understand how the underlying feature distributions evolve over time, which can provide insights into potential shifts in the data. The unsupervised characterization provides a way to explore the data, allowing to investigate changes in the prior, marginal or conditional distributions over time and sources. This can highlight key temporal subgroups, reveal whether the underlying distributions are stable, and inform decisions about how to best approach supervised model training.

### 5.1 Prior probability shift
First, we are going to assess the prior probability shift. This means tracking the evolution of the GDM label across monthly batches. To do it, we are going to use the `estimate_univariate_data_temporal_map` function, which estimates a `DataTemporalMap` object from a `DataFrame` containig individuals in rows and the variables in columns, being one of these columns the analysis date in `date` format. The function returns a `DataTemporalMap` object or a dictionary of `DataTemporalMap` objects depending on the number of analysis variables, which summarizes the temporal trajectory. Then, you can plot the results using the `plot_univariate_data_temporal_map` function, which displays a heatmap or a time series plot of a `DataTemporalMap` object.


In [ ]:
# Estimate the DataTemporalMap object for each variable
univariate_dtm = ds.estimate_univariate_data_temporal_map(
    data=dataset_formatted,
    date_column_name=DATE_COLUMN,
    period=PERIOD,
    verbose=True
)

In [ ]:
import plotly.io as pio
pio.renderers.default = "jupyterlab"

In [ ]:
# Display the series of the GDM label
univariate_dtm_plot = ds.plot_univariate_data_temporal_map(
    data_temporal_map=univariate_dtm[LABEL],
    absolute=False,
    log_transform=False,
    sorting_method='frequency',
    mode='series'
)
univariate_dtm_plot.show()

The result shows no clue of prior probability shift. However, there is a marked class imbalance, with the negative class consistenly more prevalent than the positive class. We also observe ta clear seasonality in GDM prevalence, with a higher rates during the warm months.

### 5.2 Covariate shift
After ruling out prior probability shift, the next step is to assess the covariate shift. We can use the `estimate_multivariate_data_temporal_map` function on a tidy `DataFrame`. This function performs a multivariate analysis applying a dimensionality reduction method (e.g., PCA, MCA, FAMD) to obtain a low-dimensional representation of the original data in 2 or 3 dimensions. The function returns a `MultivariateDataTemporalMap` object with the result of the study. Visualize the results with  `plot_multivariate_data_temporal_map`, which renders a density heatmap for each retained dimension. Interpreting these plots is straightforward: systematic left/right shifts or widening/narrowing bands across months indicate drift in location or scale of p(x); flat, horizontally stable bands indicate no covariate shift.


In [ ]:
# Drop the label column from the data to perform a covariate analysis
dataset_without_label = dataset_formatted.drop(columns=[LABEL])

REDUCTION_METHOD = 'PCA'
DIMENSIONS = 3
# Perform the multivariate covariate analysis
multivariate_dtm = ds.estimate_multivariate_data_temporal_map(
    data=dataset_without_label,
    date_column_name=DATE_COLUMN,
    kde_resolution=20,
    dimensions=DIMENSIONS,
    period=PERIOD,
    dim_reduction=REDUCTION_METHOD,
    scatter_plot=False,
    scale=True,
    verbose=True
)

In [ ]:
# Plot the temporal heatmaps of the first 3 prinipal components
multivariate_dtm_plot = ds.plot_multivariate_data_temporal_map(
    data_temporal_map=multivariate_dtm
)
multivariate_dtm_plot.show()

The heatmaps for PC1-PC3 remain mostly stable across the entire period, showing no clear evidence of covariate shift.

##### Information Geometric Temporal (IGT) projections
To fully assess the covariate shift in this dataset, as well as temporal subgroups and trends in the covariates, we can obtain the IGT projections with the `estimate_IGT_projection` function. The IGT projection is a technique to visualize the temporal relationships between data batches by projecting the data into a lower-dimensional space (e. g., 2D or 3D), with time batches represented as points. The distance between points reflects the probabilistic distance between the data distributions of those time batches. This function retunrs an `IGTProjection` object that can be visualized with the `plot_IGT_projection` function.

In [ ]:
# Estimate the IGT projection from the multivariate data temporal map object
multi_igt = ds.estimate_igt_projection(
        data_temporal_map=multivariate_dtm,
        dimensions=DIMENSIONS
    )

In [ ]:
# Plot the IGT projection
multi_igt_plot = ds.plot_IGT_projection(
    multi_igt,
    dimensions=DIMENSIONS,
    trajectory=True,
    color_palette='Spectral'
)
multi_igt_plot.show()

The covariates' IGT projection plot exhibits a light temporal trend over months, with a seasonal component that is accentuated from January 2023.

### 5.3 Concept shift
After assessing the covariate shift, the next step is to study the presence of concept shift. The `estimate_conditional_data_temporal_map` function allows to perform conditional distribution study with dimensionality reduction (e.g., PCA, MCA, FAMD) in 1, 2 or 3 dimension. The function returns a dictionary where the keys are the labels in the dataset, and the values are `MultiVariateDataTemporalMap` objects representing the temporal maps generated for each labels. Interpreting these plots is straightforward: systematic left/right shifts or widening/narrowing bands across months indicate drift in location or scale of p(x|y); flat, horizontally stable bands indicate no concept shift.

In [ ]:
# Estimate the multivariate conditional variability analysis
conditional_dtm = ds.estimate_conditional_data_temporal_map(
    data=dataset_formatted,
    date_column_name=DATE_COLUMN,
    label_column_name=LABEL,
    kde_resolution=20,
    dimensions=3,
    period=PERIOD,
    dim_reduction=REDUCTION_METHOD,
    scale=True,
    scatter_plot=False,
    verbose=True
)

In [ ]:
# Plot the temporal heatmaps of the first 3 principal components
conditional_dtm_plot_list = ds.plot_conditional_data_temporal_map(
    data_temporal_map_dict=conditional_dtm
)
for fig in conditional_dtm_plot_list:
    fig.show()

The conditional heatmaps show some noticeable temporal variability. Starting with the heatmaps of the PC1, both labels, but especially the positive class, show a change in the distribution around October 2022. We can observe how the distribution disperses over a few months and then concentrates again around a higher value than in previous periods. In the negative class plot, we observe a more disperse distribution in early months. In the PC2 plot we can also observe some changes in the conditional distribution. First, in the negative class distribution, the mean shifts slightly downward over time. In the positive class distribution, the mean also shifts slightly downward over time and there is also some dispersion over December 2022. These results suggest the presence of a concept shift in the data. In order to know more about the source of this variability, we are going to do an univariate conditional study.

In [ ]:
# Separate data by label for conditional analysis
dataset_formatted_0 = dataset_formatted[dataset_formatted[LABEL] == '0']
dataset_formatted_1 = dataset_formatted[dataset_formatted[LABEL] == '1']

# Estimate the data temporal maps for each conditional subset
univariate_dtm_0 = ds.estimate_univariate_data_temporal_map(
        data=dataset_formatted_0,
        date_column_name=DATE_COLUMN,
        period=PERIOD,
        verbose=True
    )
univariate_dtm_1 = ds.estimate_univariate_data_temporal_map(
        data=dataset_formatted_1,
        date_column_name=DATE_COLUMN,
        period=PERIOD,
        verbose=True
    )

In [ ]:
# We select the feature to do the conditional study
FEATURE = 'fasting_glucose_mgdl'

# Plot the data temporal map of the negative class
univariate_dtm_0_plot = ds.plot_univariate_data_temporal_map(
    data_temporal_map=univariate_dtm_0[FEATURE],
    mode='heatmap',
    title='Probability distribution DTH of negative GDM cases'
)
univariate_dtm_0_plot.show()

In [ ]:
# Plot the data temporal map of the positive class
univariate_dtm_1_plot = ds.plot_univariate_data_temporal_map(
    data_temporal_map=univariate_dtm_1[FEATURE],
    mode='heatmap',
    title='Probability distribution DTH of positive GDM cases'
)
univariate_dtm_1_plot.show()

We observe a marked shift over time in the conditional distribution of fasting OGTT glucose. In the early months, a substantial fraction of encounters labeled GDM-positive exhibit low fasting values (≈60–80 mg/dL)—a pattern consistent with the coding issue described in the use case. This misclassification diminishes gradually from late-2022 into early-2023 as workflows and coding are standardized. By March 2023, the fasting distribution among GDM-positive cases is re-centered at higher values and more clearly separated from the negative cohort, aligning with guideline-consistent diagnostic patterns.

##### Information Geometric Temporal (IGT) Projections
We will study the IGT projection of the conditional distributions as we did above with the covariate shift, looking for temporal changes, subgroups or trends.

In [ ]:
# Estimate the IGT projection from the multivariate conditional data temporal map dictionary
conditional_igt = ds.estimate_igt_projection(
        data_temporal_map=conditional_dtm,
        dimensions=DIMENSIONS
    )

In [ ]:
# Plot the IGT projection
conditional_IGT_plot = ds.plot_IGT_projection(
    conditional_igt,
    dimensions=DIMENSIONS,
    trajectory=True
)
conditional_IGT_plot.show()

The IGT projection of the conditional distribution clearly shows a temporal trend across monthly batches. We can distinguish 2 different temporal subgroups based on the heatmaps and IGT projection. The first one would be from January 2021 to August 2022. The second one from July 2023 to December 2024. Between both groups, we identify some transition batches. These results align with the ones obtained from the conditional heatmaps.
The IGT projection of the conditional distribution reveals a clear temporal trend across the monthly batches. This trend provides compelling evidence that the dataset exhibits a concept shift over time.

## 6 Supervised characterization
---

In this section, we explore how to apply supervised machine learning models to analyze and predict Gestational Diabetes Mellitus (GDM) in the context of dataset shifts over time. We will demonstrate the use of Random Forest classifiers to understand and predict the GDM label based on clinical features (e.g., fasting glucose, HbA1c, BMI...) while accounting for temporal dynamics using de `dashi` library. The goal is to highlight how models trained on earlier data may degrade when applied to future, and to show how time-aware methods can mitigade this degradation.

We will start by using the `estimate_multibatch_models`. This function automatically trains RandomForest based models across multiple batches (temporal or source) for both classification and regression tasks. Additionally, it validates each trained model's performance on every other batch. The function retunrs a dictionary containing the following metrics for each batch and model combination.

The classification metrics storaged are:
- 'AUC_{class_identifier}'
- 'AUC_MACRO'
- 'LOGLOSS'
- 'RECALL_{class_identifier}'
- 'PRECISION_{class_identifier}'
- 'F1-SCORE_{class_identifier}'
- 'ACCURACY'
- 'RECALL_MACRO'
- 'RECALL_MICRO'
- 'RECALL_WEIGHTED'
- 'PRECISION_MACRO'
- 'PRECISION_MICRO'
- 'PRECISION_WEIGHTED'
- 'F1-SCORE_MACRO'
- 'F1-SCORE_MICRO'
- 'F1-SCORE_WEIGHTED'

The regression metrics storaged are:
- 'MEAN_ABSOLUTE_ERROR'
- 'MEAN_SQUARED_ERROR'
- 'ROOT_MEAN_SQUARED_ERROR'
- 'R_SQUARED'
  

We will begin by training the models using a cumulative strategy, where each classifier is trained on all the available training data up to that point in time. This approach is commonly used in practice, as it allows the model to learn from the most recent data while incorporating all previously available information.

In [ ]:
numerical_input = numerical_features.copy()

# Obtain the metrics of the batched models trained with cumulative strategy
metrics_c = ds.estimate_multibatch_models(
        data=dataset_formatted,
        inputs_numerical_column_names=numerical_input,
        output_classification_column_name=LABEL,
        date_column_name=DATE_COLUMN,
        period='year',
        learning_strategy='cumulative',
    )

We can visualize the performance metrics using the `plot_multibatch_performance` function, which displays a heatmap of the specified metric where the y-axis are the training experiences and the x-axis the test experiences.

In [ ]:
# Plot the positive class recall's heatmap
recall_1_c = ds.plot_multibatch_performance(
        metrics=metrics_c,
        metric_name='RECALL_1'
    )
recall_1_c.show()

In [ ]:
# Plot the ROC-AUC MACRO's heatmap
auc_c = ds.plot_multibatch_performance(
        metrics=metrics_c,
        metric_name='AUC_MACRO'
    )
auc_c.show()

As we can observe models trained on batches up to 2022 perform uniformly poorly across all test batches. This broad failure aligns with the early labeling defect. Training on corrupted labels teaches a boundary that over-calls positives, so both AUC and F1 collapse. In contrast, models trained after the correction show a marked performance recovery.

Now, we will train the models with a from-scratch strategy, where each classifier is trained with the training data from a single temporal batch.

In [ ]:
# Obtain the metrics of the batched models trained with from-scratch strategy
metrics_fs = ds.estimate_multibatch_models(
        data=dataset_formatted,
        inputs_numerical_column_names=numerical_input,
        output_classification_column_name=LABEL,
        date_column_name=DATE_COLUMN,
        period='year',
        learning_strategy='from_scratch',
    )

In [ ]:
# Plot the positive class recall's heatmap
recall_1_fs = ds.plot_multibatch_performance(
        metrics=metrics_fs,
        metric_name='RECALL_1'
    )
recall_1_fs.show()

In [ ]:
# Plot the ROC-AUC MACRO's heatmap
auc_fs = ds.plot_multibatch_performance(
        metrics=metrics_fs,
        metric_name='AUC_MACRO'
    )
auc_fs.show()

The from-scratch training reveals similar results to the cumulative training, which closely aligns with the temporal subgroups identified in the unsupervised characterization again. This reinforces the idea that the temporal shift in the data is a key factor influencing model performance. Notably, the performance improves when models are trained and tested after the labeling issue correction. Both from-scratch and cumulative training strategies show similar performance, with the advantage that the from-scratch training is less computationally demanding.

## 7 Conclusions and practical suggestions

- Uncover the "unknown unknowns" before modeling: before you even begin training a model, an unsupervised analysis is crucial for building trust in your data. This tutorial shows that simple summary statistics might miss underlying instability. dashi's unsupervised tools, especially the Information Geometric Temporal (IGT) projections, successfully identified a hidden concept shift, clustering the data into distinct "pre-correction" and "post-correction" periods. This provides an objective, model-agnostic view of your dataset's stability over time.

- Diagnose the precise nature of the shift: dataset shift is often a vague term. dashi enables you to move from suspicion to a precise diagnosis. The tutorial effectively demonstrates how to:

  - Pinpoint the problem as a concept shift using conditional distribution analysis (p(x|y)).

  - Isolate the features most affected by the shift (in this case, fasting_glucose_mgdl).

- Quantify the clinical impact of the shift: an abstract statistical shift is interesting, but its impact on model performance is highly relevant. The supervised analysis in dashi provides a clear, quantifiable link between the data shift and its consequences. The performance heatmaps are a powerful tool to visualize exactly when and how a model's performance degrades. This transforms the problem from a data quality issue into a measurable risk to the model's reliability and utility.

- Make data-driven modeling decisions: the insights from dashi provide actionable evidence to guide your modeling strategy. The tutorial proves that a naive "train on all data" (cumulative) approach can be harmful, as the model is poisoned by outdated, incorrect data relationships. The analysis provides a strong justification for adopting a more robust strategy, such as:

  - Excluding data from the problematic period (2021-2022).

  - Training a new model from scratch on recent, reliable data.

  - Implementing a rolling-window training scheme to ensure the model adapts to the latest data regime.

## 8 Bibliography
- Andreu-Perez, J., Poon, C. C. Y., Merrifield, R. D., Wong, S. T. C., & Yang, G.-Z. (2015). Big Data for Health. IEEE Journal of Biomedical and Health Informatics, 19(4), 1193–1208. IEEE Journal of Biomedical and Health Informatics. https://doi.org/10.1109/JBHI.2015.2450362
- Gewin, V. (2016). Data sharing: An open mind on open data. Nature, 529(7584), 117–119. https://doi.org/10.1038/nj7584-117a
- Moreno-Torres, J. G., Raeder, T., Alaiz-Rodríguez, R., Chawla, N. V., & Herrera, F. (2012). A unifying view on dataset shift in classification. Pattern Recognition, 45(1), 521–530. https://doi.org/10.1016/j.patcog.2011.06.019
- Quiñonero-Candela, J. (Ed.). (2009). Dataset shift in machine learning. MIT Press.
- Sáez, C., Gutiérrez-Sacristán, A., Kohane, I., García-Gómez, J. M., & Avillach, P. (2020). EHRtemporalVariability: Delineating temporal data-set shifts in electronic health records. GigaScience, 9(8), giaa079. https://doi.org/10.1093/gigascience/giaa079
- Schlegel, D. R., & Ficheur, G. (2017). Secondary Use of Patient Data: Review of the Literature Published in 2016. Yearbook of Medical Informatics, 26(1), 68–71. https://doi.org/10.15265/IY-2017-032
